In [2]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from collections import defaultdict
import matplotlib.pyplot as plt
import os
import pickle
from utils.utils import *
from utils.comp_result import*
with open("processed_data/processed_data/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

C:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
# Ordine class-incremental
# Label map utilizzata per suddivvidere meglio il dataset e ottenere
# gruppi di dati indicativi
LABEL_MAP = {
    "BENIGN": "Normal",

    "DDoS": "DoS",
    "DoS Hulk": "DoS",
    "DoS GoldenEye": "DoS",
    "DoS slowloris": "DoS",
    "DoS Slowhttptest": "DoS",

    "PortScan": "Probe",

    "FTP-Patator": "BruteForce",
    "SSH-Patator": "BruteForce",
}



CLASS_ORDER = ["Normal", "DoS", "Probe", "BruteForce"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_ORDER)}
NUM_CLASSES = len(CLASS_ORDER)

# Loading data

DATA_DIR = "processed_data/processed_data/"

X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))

X_val = np.load(os.path.join(DATA_DIR, "X_val.npy"))
y_val = np.load(os.path.join(DATA_DIR, "y_val.npy"))

X_test = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_test = np.load(os.path.join(DATA_DIR, "y_test.npy"))

print("TRAIN:", X_train.shape, y_train.shape)
print("VAL  :", X_val.shape, y_val.shape)
print("TEST :", X_test.shape, y_test.shape)

print("Train labels:", np.unique(y_train))
print("Val labels  :", np.unique(y_val))
print("Test labels :", np.unique(y_test))

TRAIN: (3854088, 49) (3854088,)
VAL  : (428233, 49) (428233,)
TEST : (1070581, 49) (1070581,)
Train labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
Val labels  : [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
Test labels : [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]


In [4]:
strategies = ["full", "none", "memory"]
results = {}

for strat in strategies:
    print(f"\nRunning strategy: {strat}")

    history = run_experiment(
        X_train, y_train,
        X_val, y_val,
        X_test, y_test,
        label_encoder,
        strategy=strat,
        memory_size=200,
        poisoned=True,
        poison_fraction=0.2,
        balanced=True,
        epochs=10
    )

    results[strat] = history


Running strategy: full
Kept 3848086 / 3854088 samples after label filtering
Kept 427566 / 428233 samples after label filtering
Kept 1068914 / 1070581 samples after label filtering
>> Using BALANCED datasets
Normal: 16551 samples
Probe: 16551 samples
DoS: 16551 samples
BruteForce: 16551 samples
DoS: 1839 samples
Probe: 1839 samples
BruteForce: 1839 samples
Normal: 1839 samples
Normal: 4597 samples
DoS: 4597 samples
BruteForce: 4597 samples
Probe: 4597 samples

=== STEP 1 POISONED ===
Step classes: ['Normal']
X shape: (16551, 49)
y distribution: (array([0]), array([16551]))
Step classes: ['Normal']
X shape: (1839, 49)
y distribution: (array([0]), array([1839]))
Step classes: ['Normal']
X shape: (4597, 49)
y distribution: (array([0]), array([4597]))
Accuracy Normal: 0.964

=== STEP 2 POISONED ===
Step classes: ['Normal', 'DoS']
X shape: (33102, 49)
y distribution: (array([0, 1]), array([16551, 16551]))
Step classes: ['Normal', 'DoS']
X shape: (3678, 49)
y distribution: (array([0, 1]), ar

In [12]:
class_order = CLASS_ORDER
total_step = len(CLASS_ORDER)
for strat in strategies:
    padded_result[strat] = pad__history(results[strat], class_order, total_step)

In [13]:
print(padded_result)

{'Normal': [nan, nan, nan, nan], 'DoS': [nan, nan, nan, nan], 'Probe': [nan, nan, nan, nan], 'BruteForce': [nan, nan, nan, nan], 'full': {'Normal': [0.963889493147705, 0.9375679791168153, 0.5344790080487274, 0.3304328910158799], 'DoS': [nan, 0.852077441809876, 0.997607135088101, 0.9986948009571459], 'Probe': [nan, nan, 0.0, 0.0], 'BruteForce': [nan, nan, nan, 0.0]}, 'none': {'Normal': [1.0, 1.0, 0.9963019360452469, 0.9821622797476616], 'DoS': [nan, 0.0026103980857080703, 0.517076354144007, 0.7056776158364151], 'Probe': [nan, nan, 0.0, 0.0], 'BruteForce': [nan, nan, nan, 0.0]}, 'memory': {'Normal': [1.0, 1.0, 0.9693278224929301, 0.9343049815096802], 'DoS': [nan, 0.0, 0.7054600826626061, 0.7326517293887318], 'Probe': [nan, nan, 0.0, 0.0], 'BruteForce': [nan, nan, nan, 0.0]}}


{'full': {'Normal': [0.963889493147705, 0.9375679791168153, 0.5344790080487274, 0.3304328910158799], 'DoS': [0.852077441809876, 0.997607135088101, 0.9986948009571459], 'Probe': [0.0, 0.0], 'BruteForce': [0.0]}, 'none': {'Normal': [1.0, 1.0, 0.9963019360452469, 0.9821622797476616], 'DoS': [0.0026103980857080703, 0.517076354144007, 0.7056776158364151], 'Probe': [0.0, 0.0], 'BruteForce': [0.0]}, 'memory': {'Normal': [1.0, 1.0, 0.9693278224929301, 0.9343049815096802], 'DoS': [0.0, 0.7054600826626061, 0.7326517293887318], 'Probe': [0.0, 0.0], 'BruteForce': [0.0]}}
